In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import platform
from typing import Tuple, List

# 設定繪圖風格
sns.set_theme(style="whitegrid")

def get_chinese_font() -> str:
    """
    根據作業系統自動選擇中文字型，避免亂碼。
    """
    system = platform.system()
    if system == "Windows":
        return "Microsoft JhengHei"
    elif system == "Darwin":  # macOS
        return "Arial Unicode MS"
    else:
        return "sans-serif" # Linux 常需另外設定，此為 fallback

from pathlib import Path
parent_dir = Path("test2.ipynb").resolve().parent / "result"

def prepare_data() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    讀取並預處理資料，進行合併與清洗。
    Returns:
        df_jobs, df_salary_merged, df_tools_merged
    """
    # 1. 讀取資料
    df_jobs = pd.read_csv(parent_dir / 'all_job_name.csv') # 假設此表已有 category 欄位 (前一步驟產出)
    df_exact_salary = pd.read_csv(parent_dir / 'all_exact_salary.csv')
    df_specialtys = pd.read_csv(parent_dir / 'all_specialtys.csv')

    # 2. 處理薪資資料
    # 計算平均月薪: (Min + Max) / 2
    df_exact_salary['avg_salary'] = (df_exact_salary['salaryMin'] + df_exact_salary['salaryMax']) / 2
    
    # 合併職缺與薪資 (Inner Join，只保留有明確薪資的職缺)
    df_salary_merged = pd.merge(df_jobs, df_exact_salary, on='_id', how='inner')
    
    # 排除 "Others" 與 "Unknown" 以保持圖表乾淨
    df_salary_merged = df_salary_merged[~df_salary_merged['category'].isin(['Others', 'Unknown'])]

    # 3. 處理技能資料
    # 合併職缺與技能
    df_tools_merged = pd.merge(df_jobs, df_specialtys, on='_id', how='inner')
    
    return df_jobs, df_salary_merged, df_tools_merged

def plot_charts(df_jobs: pd.DataFrame, df_salary: pd.DataFrame, df_tools: pd.DataFrame):
    """
    繪製三張核心分析圖表
    """
    font_name = get_chinese_font()
    plt.rcParams['font.sans-serif'] = [font_name]
    plt.rcParams['axes.unicode_minus'] = False # 解決負號顯示問題

    fig = plt.figure(figsize=(20, 18))
    fig.suptitle('AI 與軟體工程師職缺市場分析報告', fontsize=24, fontweight='bold', y=0.95)

    # --- 圖表 1: 各類別職缺數量分佈 (Market Demand) ---
    ax1 = plt.subplot(3, 1, 1)
    
    # 計算數量並排序
    order = df_jobs['category'].value_counts().index
    sns.countplot(data=df_jobs, y='category', order=order, palette='viridis', ax=ax1)
    
    ax1.set_title('各職務類別市場需求量 (Job Openings by Category)', fontsize=16)
    ax1.set_xlabel('職缺數量', fontsize=12)
    ax1.set_ylabel('')
    ax1.bar_label(ax1.containers[0], padding=3)

    # --- 圖表 2: 各類別薪資分佈箱型圖 (Salary Distribution) ---
    ax2 = plt.subplot(3, 1, 2)
    
    # 根據薪資中位數排序類別
    sorted_cats = df_salary.groupby('category')['avg_salary'].median().sort_values(ascending=False).index
    
    sns.boxplot(data=df_salary, x='category', y='avg_salary', order=sorted_cats, palette='coolwarm', ax=ax2)
    
    ax2.set_title('各職務類別月薪分佈 (Monthly Salary Distribution - TWD)', fontsize=16)
    ax2.set_ylabel('平均月薪 (TWD)', fontsize=12)
    ax2.set_xlabel('')
    ax2.tick_params(axis='x', rotation=15) # 旋轉標籤以免重疊
    
    # 標註中位數
    medians = df_salary.groupby('category')['avg_salary'].median()
    vertical_offset = df_salary['avg_salary'].median() * 0.05
    for xtick in ax2.get_xticks():
        cat_name = sorted_cats[xtick]
        if cat_name in medians:
            ax2.text(xtick, medians[cat_name] + vertical_offset, f'{int(medians[cat_name]):,}', 
                     horizontalalignment='center', size='small', color='black', weight='semibold')

    # --- 圖表 3: 前 15 大熱門技術 (Top Skills) ---
    ax3 = plt.subplot(3, 1, 3)
    
    top_skills = df_tools['description'].value_counts().head(15)
    sns.barplot(x=top_skills.values, y=top_skills.index, palette='magma', ax=ax3)
    
    ax3.set_title('市場需求最高的 Top 15 關鍵技術 (Most Demanded Hard Skills)', fontsize=16)
    ax3.set_xlabel('出現頻次', fontsize=12)
    ax3.bar_label(ax3.containers[0], padding=3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # 調整佈局避免標題被遮擋
    plt.show()

# 執行主程序
if __name__ == "__main__":
    try:
        # 注意：這裡假設您已經執行過上一段分類程式碼，並產生了帶有 category 的 dataframe
        # 若是直接執行，請確保 'category' 欄位存在於 df_jobs 中
        
        # 這裡為了演示，我們重新呼叫上一段邏輯或直接使用處理好的變數
        # 假設資料已在記憶體中，或是讀取您上傳後的檔案
        
        # 模擬讀取流程 (實際使用時請取消註解)
        df_j, df_s, df_sk = prepare_data()
        plot_charts(df_j, df_s, df_sk)
        
    except Exception as e:
        print(f"繪圖時發生錯誤: {e}")
        print("建議檢查：CSV 檔案路徑是否正確？是否有安裝 seaborn？中文字型是否存在？")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import os
import shutil
import platform
import matplotlib
from typing import Optional

# --- 0. Environment Cleanup (Optional but helpful) ---
def clear_matplotlib_cache():
    cache_dir = matplotlib.get_cachedir()
    print(f"Checking Matplotlib cache: {cache_dir}")
    if os.path.exists(cache_dir):
        try:
            shutil.rmtree(cache_dir)
            print("✅ Cache cleared. If fonts fail, restart kernel.")
        except Exception as e:
            print(f"⚠️ Cache clear failed: {e}")

clear_matplotlib_cache()

# --- 1. Robust Font Handling ---
def configure_chinese_font(custom_font_path: str = "NotoSansTC-Regular.ttf") -> str:
    """
    配置 Matplotlib 中文字體，優先使用本地檔案，其次系統 fallback。
    """
    # 1. Reset params first via Seaborn (Crucial step)
    sns.set_theme(style="whitegrid", rc={"axes.unicode_minus": False}) 

    font_name = None

    # 2. Strategy A: Local File
    if os.path.exists(custom_font_path):
        try:
            fm.fontManager.addfont(custom_font_path)
            prop = fm.FontProperties(fname=custom_font_path)
            font_name = prop.get_name()
            print(f"✅ Loaded local font: {font_name}")
        except Exception as e:
            print(f"⚠️ Local font load failed: {e}")

    # 3. Strategy B: System Fallback
    if not font_name:
        system = platform.system()
        if system == "Windows":
            font_name = "Microsoft JhengHei"
        elif system == "Darwin": # macOS
            font_name = "Arial Unicode MS"
        else: # Linux / Cloud
            font_name = "WenQuanYi Micro Hei" # Cloud environments often have this
        print(f"ℹ️ Using system fallback font: {font_name}")

    # 4. Apply Globally
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = [font_name] + plt.rcParams['font.sans-serif']
    
    return font_name

# Initialize Font Logic
current_font = configure_chinese_font("NotoSansTC-Regular.ttf")

# ... 接下來接您的 Data Loading (Section 2) ...
# --- 2. Data Loading & Processing ---
# Classification Logic
def classify_job_title(title):
    if not isinstance(title, str):
        return "Unknown"
    title_lower = title.lower()
    if any(kw in title_lower for kw in ['ai', 'artificial intelligence', 'machine learning', 'deep learning', 'computer vision', 'nlp', 'algorithm', '人工智慧', '機器學習', '演算法', '深度學習', '影像識別', '自然語言', 'llm', 'gpt']):
        return "AI Engineer / Researcher"
    if any(kw in title_lower for kw in ['scientist', 'analyst', 'analysis', 'mining', '資料科學', '數據分析', '資料分析']):
        return "Data Scientist / Analyst"
    if any(kw in title_lower for kw in ['data engineer', 'etl', 'big data', 'spark', 'hadoop', 'pipeline', '資料工程', '數據工程']):
        return "Data Engineer"
    if any(kw in title_lower for kw in ['devops', 'sre', 'site reliability', 'cloud', 'aws', 'gcp', 'azure', 'kubernetes', 'docker', 'cicd', '雲端', '系統工程', '運維']):
        return "DevOps / SRE / Cloud Engineer"
    if any(kw in title_lower for kw in ['qa', 'test', 'automation', 'sdet', '測試', '自動化', '品保']):
        return "QA / Automation Engineer"
    if any(kw in title_lower for kw in ['firmware', 'embedded', 'driver', 'fpga', '韌體', '嵌入式', '驅動']):
        return "Embedded / Firmware Engineer"
    if any(kw in title_lower for kw in ['fullstack', 'full-stack', '全端']):
        return "Fullstack Engineer"
    if any(kw in title_lower for kw in ['frontend', 'front-end', 'react', 'vue', 'angular', 'javascript', 'html', 'ui', 'ux', 'web', '前端']):
        return "Frontend Engineer"
    if any(kw in title_lower for kw in ['backend', 'back-end', 'server', 'php', 'java', 'golang', 'ruby', 'node', 'c#', '.net', 'django', 'flask', 'spring', '後端']):
        return "Backend Engineer"
    if any(kw in title_lower for kw in ['software engineer', 'developer', 'programmer', 'python', 'c++', 'engineer', '工程師', '軟體']):
        return "General Software Engineer"
    return "Others"

# Load Data
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ... (保留你原本的 classify_job_title 函式) ...

# --- 請從這裡開始替代 ---
# --- 2. Data Loading & Processing ---

# Load Data (假設路徑不變)
df_jobs = pd.read_csv(r'C:\Users\kionc\Projects\AIPE02_Tutorial\Taiwan_Job_Market_Analysis\result\all_job_name.csv')
df_skills = pd.read_csv(r"C:\Users\kionc\Projects\AIPE02_Tutorial\Taiwan_Job_Market_Analysis\result\all_specialtys.csv")
# Merge
df_merged = pd.merge(df_jobs, df_skills, on='_id', how='inner')
df_merged = df_merged[df_merged['category'] != 'Others']

# 【關鍵步驟 1】計算每個類別的職缺總數 (N)
# 使用 nunique() 確保計算的是「職缺數」而非「技能出現總次數」
# 【Logic Validation: Correct】
# 計算每個類別的職缺總數 (N)
category_stats = df_merged.groupby('category')['_id'].nunique().reset_index(name='total_jobs')

# 計算技能頻次與排名
skill_counts = df_merged.groupby(['category', 'description']).size().reset_index(name='count')
skill_counts['rank'] = skill_counts.groupby('category')['count'].rank(method='first', ascending=False)

# Filter Top 3 & Convert Rank
top_3_skills = skill_counts[skill_counts['rank'] <= 3].copy()
top_3_skills['rank'] = top_3_skills['rank'].astype(int)

# Merge N count back
top_3_skills = pd.merge(top_3_skills, category_stats, on='category', how='left')

# Create Labels
top_3_skills['category_label'] = top_3_skills.apply(
    lambda x: f"{x['category']}\n(n={x['total_jobs']})", axis=1
)

# Sorting: By Category Alphabetical (as per your code)
top_3_skills = top_3_skills.sort_values(['category', 'rank'])
labels_order = sorted(top_3_skills['category_label'].unique())
# --- 3. Plotting (Horizontal with N count) ---
# --- 3. Plotting (Refined: No Y-Axis Title) ---

plt.figure(figsize=(15, 12)) 

blue_gradient_palette = ["#08519c", "#3182bd", "#6baed6"]

# 建立圖表物件
ax = sns.barplot(
    data=top_3_skills, 
    y='category_label', 
    x='count', 
    hue='rank', 
    order=labels_order,
    hue_order=[1, 2, 3], 
    palette=blue_gradient_palette,
    orient='h'
)

# Annotations Logic (維持原樣)
for rank_idx, container in enumerate(ax.containers):
    current_rank = rank_idx + 1 
    rank_data = top_3_skills[top_3_skills['rank'] == current_rank].set_index('category_label')
    
    for i, bar in enumerate(container):
        cat_label = labels_order[i]
        if cat_label in rank_data.index:
            skill_name = rank_data.loc[cat_label, 'description']
            count_val = rank_data.loc[cat_label, 'count']
            
            width = bar.get_width()
            y_pos = bar.get_y()
            height = bar.get_height()
            
            if width > 0:
                ax.text(
                    width + 2,
                    y_pos + height / 2, 
                    f"{skill_name} ({int(count_val)})", 
                    va='center', 
                    ha='left', 
                    fontsize=10, 
                    fontweight='bold', 
                    color='#2c3e50',
                    fontfamily=current_font 
                )

# --- 設定標題與標籤 ---
plt.title('Top 3 Core Tools per Job Category', fontsize=20, fontweight='bold', pad=20)
plt.xlabel('Tool Frequency Count', fontsize=14)

# 【關鍵修正】：將 Y 軸標題設為 None 或空字串
plt.ylabel(None) 
# 或者使用: ax.set_ylabel('')

plt.yticks(fontsize=11)
plt.legend(title='Rank', loc='lower right')
plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
# plt.savefig('chart4_horizontal_with_n_no_ylabel.png', dpi=150)
print("✅ Chart generated successfully (Y-axis title removed).")